# 06 - Final Head-to-Head Benchmark Evaluation (PPO vs Baselines)

This notebook evaluates the trained **PPO Policy** directly against **FIFO, SJF, Priority, and Best-Fit** across all **6 AI cluster scenarios** on held-out unseen test seeds (`[501, 602, 703]`).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is in python path
sys.path.insert(0, os.path.abspath('..'))

from baselines.fifo import FIFOScheduler
from baselines.sjf import SJFScheduler
from baselines.priority import PriorityScheduler
from baselines.best_fit import BestFitScheduler
from evaluation.ppo_policy import PPOPolicyScheduler
from evaluation.evaluator import Evaluator
from evaluation.reports import generate_comparison_dataframe
from workloads.scenarios import list_scenarios

print("All evaluation modules imported successfully!")

## 1. Execute Benchmark Across All 5 Schedulers

In [ ]:
test_seeds = [501, 602, 703]
scenarios = ["balanced", "training_heavy", "short_job_heavy", "bursty", "gpu_fragmentation", "high_load"]

policies = [
    FIFOScheduler(),
    SJFScheduler(),
    PriorityScheduler(),
    BestFitScheduler(),
    PPOPolicyScheduler(checkpoint_path="../checkpoints/ppo_final.pt", cluster_config_path="../configs/cluster_small.yaml"),
]

evaluator = Evaluator(cluster_config_path="../configs/cluster_small.yaml", horizon_seconds=3600.0)

all_results = {}
for pol in policies:
    print(f"Evaluating [{pol.name}] on all {len(scenarios)} scenarios...")
    all_results[pol.name] = evaluator.evaluate_scheduler(pol, scenarios=scenarios, seeds=test_seeds)

print("\nHead-to-head benchmark complete!")

## 2. Comparative Benchmark Tables per Scenario

In [ ]:
for sc in scenarios:
    df_sc = generate_comparison_dataframe(all_results, scenario_name=sc)
    print(f"\n{'='*65}")
    print(f"Scenario: {sc.upper()}")
    print(f"{'='*65}")
    print(df_sc.to_string(index=False))

## 3. Visual Comparisons Across Schedulers

In [ ]:
pol_names = [p.name for p in policies]
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B3', '#CCB974']

fig, axes = plt.subplots(2, 2, figsize=(15, 9))

# 1. Mean JCT in GPU Fragmentation
jcts = [all_results[p]["gpu_fragmentation"]["summary"]["mean_jct"]["mean"] for p in pol_names]
axes[0, 0].bar(pol_names, jcts, color=colors)
axes[0, 0].set_title("Mean Job Completion Time: GPU Fragmentation (Lower is better)")
axes[0, 0].set_ylabel("Seconds")
axes[0, 0].grid(True, alpha=0.3)

# 2. GPU Utilization in Balanced
utils = [all_results[p]["balanced"]["summary"]["gpu_utilization"]["mean"] * 100 for p in pol_names]
axes[0, 1].bar(pol_names, utils, color=colors)
axes[0, 1].set_title("Cluster GPU Utilization: Balanced Scenario (Higher is better)")
axes[0, 1].set_ylabel("Utilization (%)")
axes[0, 1].grid(True, alpha=0.3)

# 3. Queue Wait Time in Training Heavy
waits = [all_results[p]["training_heavy"]["summary"]["mean_wait_time"]["mean"] for p in pol_names]
axes[1, 0].bar(pol_names, waits, color=colors)
axes[1, 0].set_title("Mean Queue Wait: Training Heavy (Lower is better)")
axes[1, 0].set_ylabel("Seconds")
axes[1, 0].grid(True, alpha=0.3)

# 4. Cumulative Reward in High Load
rewards = [all_results[p]["high_load"]["summary"]["cumulative_reward"]["mean"] for p in pol_names]
axes[1, 1].bar(pol_names, rewards, color=colors)
axes[1, 1].set_title("Cumulative Reward: High Load (Higher is better)")
axes[1, 1].set_ylabel("Reward")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()